In [1]:
import os
import numpy as np
import pandas as pd
import librosa 
import soundfile as sf
import random
from pathlib import Path

In [2]:
## Sample rate
SAMPLING_RATE=16000
## Signal to noise ratio in db more db good speech
SNR_LEVELS=[0,5,15,10]

SPEECH_FOLDER='16K-LP7'
NOISE_FOLDER='BLOWER NOISE'

## New folder to save the data
OUTPUT_FOLDER='noisy_dataset'

## Creating a new directory to save the generated data in
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [3]:
## Normalization function
def normalize_audio(audio):
    ## Converts each value of amplitude into positive then finds the max value in the whole audio file
    max_val=np.max(np.abs(audio))

    ## if the value of max is 0, the audio file is empty
    if max_val==0:
        return audio
    
    ## Normalizes the audio amplitude between 0 to 1
    return audio/max_val

In [4]:
## SNR Adjustment function
def adjus_noise_for_snr(speech, noise, snr_db):

    ## Finds the power of the speech
    speech_power=np.mean(speech**2)

    ## Finds the power of the noise 
    noise_power=np.mean(noise**2)
    if noise_power==0:
        return 0

    ## we set a signal to noise ratio manually to have variety in noise levels in speech
    target_noise_power=speech_power/(10**(snr_db/10))

    ## multipling square root of power with amplitude of noise to get target snr
    noise=noise*np.sqrt(target_noise_power/noise_power)
    return noise

In [5]:
## Loading files
speech_files=list(Path(SPEECH_FOLDER).rglob('*.wav'))
noise_files=list(Path(NOISE_FOLDER).rglob('*.mp3'))

print('Total speech files:',len(speech_files))
print('Total  noise files:',len(noise_files))

Total speech files: 1444
Total  noise files: 4


In [8]:
## Dataset Generation
for speech_file in speech_files:
    ## Load speech
    speech,_=librosa.load(speech_file,sr=SAMPLING_RATE)

    ## Normalize speech
    speech=normalize_audio(speech)

    ## Choose random noise
    noise_file=random.choice(noise_files)

    ## Load noise
    noise, _= librosa.load(noise_file,sr=SAMPLING_RATE)

    ## Repeat noise if shorter than speech
    if len(noise)< len(speech):
        ## np.ceil() rounds the numbers to the nearest integer
        repeat_times=int(np.ceil(len(speech)/len(noise)))

        ## np.tile() repeats the same array into repeat times and arranges into one big array
        noise=np.tile(noise, repeat_times)

    ## Random crop noise
    ## cuts the extra part only useful noises+speech remains
    start=random.randint(0,len(noise)-len(speech))
    noise=noise[start:start+len(speech)]

    ## Random snr choosing
    snr=random.choice(SNR_LEVELS)

    ## Adjust noise levels
    noise=adjus_noise_for_snr(speech, noise, snr)
    

    ## Mix speech + noise
    noisy = speech + noise

    ## Prevent clipping but preserve energy ratio between clean and noisy
    max_val = np.max(np.abs(noisy))
    if max_val > 1.0:
        noisy = noisy / max_val
        speech = speech / max_val # MUST SCALE CLEAN SPEECH BY SAME FACTOR

    ## Output names
    speech_name = Path(speech_file).stem
    noise_name = Path(noise_file).stem
    
    output_name = f'{speech_name}_snr{snr}_noise{noise_name}.wav'
    noisy_output_path = os.path.join(OUTPUT_FOLDER, output_name)
    
    ## Save noisy audio
    sf.write(noisy_output_path, noisy, SAMPLING_RATE)
    
    ## CRITICAL: Save the strictly aligned clean audio
    aligned_clean_folder = 'aligned_clean_dataset'
    os.makedirs(aligned_clean_folder, exist_ok=True)
    clean_output_path = os.path.join(aligned_clean_folder, output_name)
    sf.write(clean_output_path, speech, SAMPLING_RATE)


print('DATASET GENERATION COMPLETED SUCCESSFULLY')


DATASET GENERATION COMPLETED SUCCESSFULLY


In [9]:
noisy_data=list(Path(OUTPUT_FOLDER).rglob('*.wav'))

print('Total noisy data files:',len(noisy_data))

Total noisy data files: 2783
